# 🌦️ Crawl Weather Data 2022-2024 với Class Balancing cho XGBoost

**Mục tiêu:**
- Crawl dữ liệu thời tiết 3 năm (2022, 2023, 2024)
- Xử lý class imbalance bằng các kỹ thuật:
  - SMOTE (Synthetic Minority Over-sampling)
  - Random Under-sampling cho majority class
  - Class weights trong XGBoost
- Tạo dataset cân bằng để train XGBoost chính xác hơn

**Output:** `./data/weather_balanced_2022_2024.csv`

In [ ]:
# Import libraries
import requests
import pandas as pd
from datetime import datetime, timedelta
import time
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from google.colab import files  # For file download

# Create data directory
os.makedirs('./data', exist_ok=True)

print("✅ Libraries imported successfully")
print("📂 Output directory: ./data/")

## Bước 1: Cấu hình crawling 3 năm (2022-2024)

In [ ]:
# CSV output path
OUTPUT_CSV = './data/weather_raw_2022_2024.csv'
BALANCED_CSV = './data/weather_balanced_2022_2024.csv'

# OpenWeatherMap API Key
OPENWEATHER_API_KEY = "YOUR_OPENWEATHER_API_KEY"

# Define ALL 63 cities
CITIES = [
    {"id": 1559969, "name": "Nghệ An", "lat": 19.33333, "lon": 104.833328},
    {"id": 1559970, "name": "Ninh Bình", "lat": 20.25, "lon": 105.833328},
    {"id": 1559971, "name": "Ninh Thuận", "lat": 11.75, "lon": 108.833328},
    {"id": 1559972, "name": "Sóc Trăng", "lat": 9.66667, "lon": 105.833328},
    {"id": 1559975, "name": "Trà Vinh", "lat": 9.83333, "lon": 106.25},
    {"id": 1559976, "name": "Tuyên Quang", "lat": 22.116671, "lon": 105.25},
    {"id": 1559977, "name": "Vĩnh Long", "lat": 10.16667, "lon": 106.0},
    {"id": 1559978, "name": "Yên Bái", "lat": 21.5, "lon": 104.666672},
    {"id": 1562412, "name": "Lào Cai", "lat": 22.33333, "lon": 104.0},
    {"id": 1564676, "name": "Tiền Giang", "lat": 10.41667, "lon": 106.166672},
    {"id": 1565033, "name": "Thừa Thiên-Huế", "lat": 16.33333, "lon": 107.583328},
    {"id": 1565088, "name": "Kon Tum", "lat": 14.75, "lon": 107.916672},
    {"id": 1566165, "name": "Thanh Hóa", "lat": 20.0, "lon": 105.5},
    {"id": 1566338, "name": "Thái Bình", "lat": 20.5, "lon": 106.333328},
    {"id": 1566557, "name": "Tây Ninh", "lat": 11.33333, "lon": 106.166672},
    {"id": 1567643, "name": "Sơn La", "lat": 21.16667, "lon": 104.0},
    {"id": 1568733, "name": "Quảng Trị", "lat": 16.75, "lon": 107.0},
    {"id": 1568758, "name": "Quảng Ninh", "lat": 21.25, "lon": 107.333328},
    {"id": 1568769, "name": "Quảng Ngãi", "lat": 15.0, "lon": 108.666672},
    {"id": 1568839, "name": "Quảng Bình", "lat": 17.5, "lon": 106.333328},
    {"id": 1569805, "name": "Phú Yên", "lat": 13.16667, "lon": 109.166672},
    {"id": 1572594, "name": "Hòa Bình", "lat": 20.33333, "lon": 105.25},
    {"id": 1575788, "name": "Long An", "lat": 10.66667, "lon": 106.166672},
    {"id": 1576632, "name": "Lạng Sơn", "lat": 21.75, "lon": 106.5},
    {"id": 1577882, "name": "Lâm Đồng", "lat": 11.5, "lon": 108.333328},
    {"id": 1579008, "name": "Kiên Giang", "lat": 10.0, "lon": 105.166672},
    {"id": 1579634, "name": "Khánh Hòa", "lat": 12.33333, "lon": 109.0},
    {"id": 1580578, "name": "TP. Hồ Chí Minh", "lat": 10.83333, "lon": 106.666672},
    {"id": 1580700, "name": "Hà Tĩnh", "lat": 18.33333, "lon": 105.75},
    {"id": 1581030, "name": "Hà Giang", "lat": 22.75, "lon": 105.0},
    {"id": 1581088, "name": "Gia Lai", "lat": 13.75, "lon": 108.25},
    {"id": 1581129, "name": "Hà Nội", "lat": 21.116671, "lon": 105.883331},
    {"id": 1581188, "name": "Cần Thơ", "lat": 9.83333, "lon": 105.666672},
    {"id": 1581297, "name": "Hải Phòng", "lat": 20.83333, "lon": 106.583328},
    {"id": 1581882, "name": "Bình Thuận", "lat": 11.08333, "lon": 108.0},
    {"id": 1582562, "name": "Đồng Tháp", "lat": 10.66667, "lon": 105.666672},
    {"id": 1582720, "name": "Đồng Nai", "lat": 11.0, "lon": 107.166672},
    {"id": 1584169, "name": "Đắk Lắk", "lat": 12.83333, "lon": 108.166672},
    {"id": 1584534, "name": "Bà Rịa - Vũng Tàu", "lat": 10.58333, "lon": 107.25},
    {"id": 1586182, "name": "Cao Bằng", "lat": 22.66667, "lon": 106.0},
    {"id": 1587871, "name": "Bình Định", "lat": 14.16667, "lon": 109.0},
    {"id": 1587974, "name": "Bến Tre", "lat": 10.16667, "lon": 106.5},
    {"id": 1594446, "name": "An Giang", "lat": 10.5, "lon": 105.166672},
    {"id": 1905099, "name": "Điện Biên", "lat": 21.33333, "lon": 102.933327},
    {"id": 1905412, "name": "Bắc Ninh", "lat": 21.08333, "lon": 106.166672},
    {"id": 1905419, "name": "Bắc Giang", "lat": 21.33333, "lon": 106.333328},
    {"id": 1905468, "name": "Đà Nẵng", "lat": 16.08333, "lon": 108.083328},
    {"id": 1905475, "name": "Bình Dương", "lat": 11.16667, "lon": 106.666672},
    {"id": 1905480, "name": "Bình Phước", "lat": 11.75, "lon": 106.916672},
    {"id": 1905497, "name": "Thái Nguyên", "lat": 21.66667, "lon": 105.833328},
    {"id": 1905516, "name": "Quảng Nam", "lat": 15.58333, "lon": 107.916672},
    {"id": 1905577, "name": "Phú Thọ", "lat": 21.33333, "lon": 105.166672},
    {"id": 1905626, "name": "Nam Định", "lat": 20.25, "lon": 106.25},
    {"id": 1905637, "name": "Hà Nam", "lat": 20.58333, "lon": 106.0},
    {"id": 1905669, "name": "Bắc Kạn", "lat": 22.16667, "lon": 105.833328},
    {"id": 1905675, "name": "Bạc Liêu", "lat": 9.25, "lon": 105.75},
    {"id": 1905678, "name": "Cà Mau", "lat": 9.08333, "lon": 105.083328},
    {"id": 1905686, "name": "Hải Dương", "lat": 20.91667, "lon": 106.333328},
    {"id": 1905699, "name": "Hưng Yên", "lat": 20.83333, "lon": 106.083328},
    {"id": 1905856, "name": "Vĩnh Phúc", "lat": 21.299999, "lon": 105.599998},
    {"id": 1584237, "name": "Đắk Nông", "lat": 12.25, "lon": 107.666672},
    {"id": 1586203, "name": "Bắc Cạn", "lat": 22.16667, "lon": 105.833328}
]

# Time range: 3 YEARS
YEAR_RANGES = [
    ("2022-01-01", "2022-12-31"),
    ("2023-01-01", "2023-12-31"),
    ("2024-01-01", "2024-12-31")
]

# 8 periods per day
PERIODS = [
    {"name": "morning_early", "hour": 0},
    {"name": "morning_late", "hour": 3},
    {"name": "noon_early", "hour": 6},
    {"name": "noon_late", "hour": 9},
    {"name": "afternoon_early", "hour": 12},
    {"name": "afternoon_late", "hour": 15},
    {"name": "evening_early", "hour": 18},
    {"name": "evening_late", "hour": 21}
]

print(f"⚙️ Configuration:")
print(f"   Cities: {len(CITIES)}")
print(f"   Years: 2022, 2023, 2024 (3 years)")
print(f"   Expected records: ~552,000")

## Bước 2: Định nghĩa crawling functions

In [ ]:
def fetch_weather_data_open_meteo(city, start_date, end_date):
    """Fetch weather from Open-Meteo API"""
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": city["lat"],
        "longitude": city["lon"],
        "start_date": start_date,
        "end_date": end_date,
        "hourly": ["temperature_2m", "relative_humidity_2m", "precipitation", "wind_speed_10m"],
        "timezone": "Asia/Bangkok"
    }
    try:
        response = requests.get(url, params=params, timeout=30)
        response.raise_for_status()
        data = response.json()
        df_hourly = pd.DataFrame({
            "city_id": city["id"],
            "city_name": city["name"],
            "time": pd.to_datetime(data["hourly"]["time"]),
            "temperature": data["hourly"]["temperature_2m"],
            "humidity": data["hourly"]["relative_humidity_2m"],
            "precipitation": data["hourly"]["precipitation"],
            "wind_speed": data["hourly"]["wind_speed_10m"]
        })
        df_hourly["date"] = df_hourly["time"].dt.date
        df_hourly["hour"] = df_hourly["time"].dt.hour
        def assign_period(hour):
            if 0 <= hour < 3: return "morning_early"
            elif 3 <= hour < 6: return "morning_late"
            elif 6 <= hour < 9: return "noon_early"
            elif 9 <= hour < 12: return "noon_late"
            elif 12 <= hour < 15: return "afternoon_early"
            elif 15 <= hour < 18: return "afternoon_late"
            elif 18 <= hour < 21: return "evening_early"
            else: return "evening_late"
        df_hourly["period"] = df_hourly["hour"].apply(assign_period)
        df_agg = df_hourly.groupby(["city_id", "city_name", "date", "period"]).agg({
            "temperature": "mean", "humidity": "mean", "precipitation": "sum", "wind_speed": "max"
        }).reset_index()
        return df_agg
    except Exception as e:
        print(f"❌ Error: {e}")
        return None

def fetch_uv_from_nasa(lat, lon, start_date, end_date):
    """Fetch UV from NASA POWER API"""
    url = "https://power.larc.nasa.gov/api/temporal/daily/point"
    params = {
        "parameters": "ALLSKY_SFC_UV_INDEX",
        "community": "RE",
        "longitude": lon,
        "latitude": lat,
        "start": start_date.replace("-", ""),
        "end": end_date.replace("-", ""),
        "format": "JSON"
    }
    try:
        response = requests.get(url, params=params, timeout=60)
        response.raise_for_status()
        data = response.json()
        uv_data = data["properties"]["parameter"]["ALLSKY_SFC_UV_INDEX"]
        dates, uv_values = [], []
        for date_str, uv_val in uv_data.items():
            date_obj = pd.to_datetime(date_str, format="%Y%m%d").date()
            dates.append(date_obj)
            uv_values.append(None if uv_val == -999 else uv_val)
        return pd.DataFrame({"date": dates, "uv_index": uv_values})
    except Exception as e:
        print(f"❌ Error: {e}")
        return None

def fetch_air_quality_from_openweather(lat, lon, start_date, end_date):
    """Fetch Air Quality from OpenWeatherMap"""
    from datetime import datetime
    start_ts = int(datetime.strptime(start_date, "%Y-%m-%d").timestamp())
    end_ts = int(datetime.strptime(end_date, "%Y-%m-%d").timestamp()) + 86399
    url = "http://api.openweathermap.org/data/2.5/air_pollution/history"
    params = {"lat": lat, "lon": lon, "start": start_ts, "end": end_ts, "appid": OPENWEATHER_API_KEY}
    try:
        response = requests.get(url, params=params, timeout=60)
        response.raise_for_status()
        data = response.json()
        if "list" not in data or len(data["list"]) == 0:
            return None
        records = []
        for item in data["list"]:
            dt = datetime.fromtimestamp(item["dt"])
            components = item["components"]
            records.append({
                "date": dt.date(), "hour": dt.hour, "aqi": item["main"]["aqi"],
                "pm2_5": components.get("pm2_5"), "pm10": components.get("pm10"),
                "co": components.get("co"), "no2": components.get("no2"),
                "o3": components.get("o3"), "so2": components.get("so2"), "nh3": components.get("nh3")
            })
        df = pd.DataFrame(records)
        def assign_period(hour):
            if 0 <= hour < 3: return "morning_early"
            elif 3 <= hour < 6: return "morning_late"
            elif 6 <= hour < 9: return "noon_early"
            elif 9 <= hour < 12: return "noon_late"
            elif 12 <= hour < 15: return "afternoon_early"
            elif 15 <= hour < 18: return "afternoon_late"
            elif 18 <= hour < 21: return "evening_early"
            else: return "evening_late"
        df["period"] = df["hour"].apply(assign_period)
        agg_df = df.groupby(["date", "period"]).agg({
            "aqi": lambda x: x.mode()[0] if len(x.mode()) > 0 else x.median(),
            "pm2_5": "mean", "pm10": "mean", "co": "mean", "no2": "mean",
            "o3": "mean", "so2": "mean", "nh3": "mean"
        }).reset_index()
        return agg_df
    except Exception as e:
        print(f"❌ Error: {e}")
        return None

print("✅ Crawling functions defined")

## Bước 3: WHO Health Risk Calculation

In [ ]:
def calculate_health_risk_score(row):
    """Calculate health risk score (0-15)"""
    score = 0
    temp = row['temperature']
    if temp < 10 or temp > 35: score += 3
    elif temp < 15 or temp > 32: score += 2
    elif temp < 18 or temp > 27: score += 1
    humidity = row['humidity']
    if humidity < 30 or humidity > 70: score += 2
    elif humidity < 40 or humidity > 60: score += 1
    uv = row.get('uv_index', 0) if pd.notna(row.get('uv_index')) else 0
    if uv > 11: score += 3
    elif uv > 8: score += 2
    elif uv > 6: score += 1
    pm25 = row.get('pm2_5', 0) if pd.notna(row.get('pm2_5')) else 0
    if pm25 > 55.5: score += 4
    elif pm25 > 37.5: score += 3
    elif pm25 > 25: score += 2
    elif pm25 > 15: score += 1
    aqi = row.get('aqi', 1) if pd.notna(row.get('aqi')) else 1
    if aqi >= 5: score += 2
    elif aqi >= 4: score += 1
    return min(score, 15)

def assign_risk_level(score):
    """Assign WHO 5-level risk"""
    if score <= 2: return "safe"
    elif score <= 5: return "low"
    elif score <= 8: return "moderate"
    elif score <= 12: return "high"
    else: return "extreme"

print("✅ Health risk functions defined")

## Bước 4: Install imbalanced-learn

In [ ]:
!pip install imbalanced-learn -q

from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from sklearn.preprocessing import LabelEncoder

print("✅ imbalanced-learn installed")

## Bước 4: Crawl dữ liệu 3 năm (2022-2024)

⚠️ **CHÚ Ý:** Process này sẽ mất ~3-6 giờ để crawl đầy đủ!
- 63 cities × 3 years × 8 periods = ~552,000 records

**Có 2 options:**
1. **Chạy cell này** để crawl (mất 3-6 giờ)
2. **Skip cell này** và upload file CSV đã crawl sẵn ở cell tiếp theo

In [ ]:
# OPTION 1: Crawl data (mất 3-6 giờ)
# Uncomment dòng dưới để chạy crawling:
# RUN_CRAWLING = True
RUN_CRAWLING = False  # Set True để crawl, False để skip

if RUN_CRAWLING:
    all_weather_data = []
    all_uv_data = []
    all_aq_data = []
    
    total_cities = len(CITIES)
    total_years = len(YEAR_RANGES)
    
    print(f"🚀 Starting crawl: {total_cities} cities × {total_years} years")
    print(f"={'='*60}")
    
    for year_idx, (start_date, end_date) in enumerate(YEAR_RANGES, 1):
        year = start_date.split('-')[0]
        print(f"\n📅 YEAR {year} [{year_idx}/{total_years}]")
        print(f"={'='*60}")
        
        for city_idx, city in enumerate(CITIES, 1):
            print(f"\n[{city_idx}/{total_cities}] {city['name']} ({year})")
            
            # 1. Weather data
            weather_df = fetch_weather_data_open_meteo(city, start_date, end_date)
            if weather_df is not None:
                all_weather_data.append(weather_df)
                print(f"   ✅ Weather: {len(weather_df)} records")
            else:
                print(f"   ❌ Weather failed")
            
            # 2. UV data
            uv_df = fetch_uv_from_nasa(city['lat'], city['lon'], start_date, end_date)
            if uv_df is not None:
                uv_df['city_id'] = city['id']
                uv_df['city_name'] = city['name']
                all_uv_data.append(uv_df)
                print(f"   ✅ UV: {len(uv_df)} days")
            else:
                print(f"   ❌ UV failed")
            
            # 3. Air Quality data
            aq_df = fetch_air_quality_from_openweather(city['lat'], city['lon'], start_date, end_date)
            if aq_df is not None:
                aq_df['city_id'] = city['id']
                aq_df['city_name'] = city['name']
                all_aq_data.append(aq_df)
                print(f"   ✅ Air Quality: {len(aq_df)} records")
            else:
                print(f"   ❌ Air Quality failed")
            
            # Courtesy delay
            time.sleep(1)
            
            # Progress update every 10 cities
            if city_idx % 10 == 0:
                print(f"\n   📊 Progress ({year}): {city_idx}/{total_cities} cities ({city_idx/total_cities*100:.1f}%)")
                print(f"   Total records so far: {sum(len(d) for d in all_weather_data):,}")
    
    print(f"\n{'='*60}")
    print(f"✅ CRAWLING COMPLETED!")
    print(f"{'='*60}")
    print(f"Weather records: {sum(len(d) for d in all_weather_data):,}")
    print(f"UV records: {sum(len(d) for d in all_uv_data):,}")
    print(f"Air Quality records: {sum(len(d) for d in all_aq_data):,}")
else:
    print("⏭️ Skipped crawling. Will use uploaded CSV file instead.")
    all_weather_data = None
    all_uv_data = None
    all_aq_data = None

## Bước 5: Merge tất cả data sources (nếu đã crawl)

In [ ]:
if all_weather_data is not None:
    print("🔗 Merging data sources...")
    
    # 1. Combine weather data
    weather_combined = pd.concat(all_weather_data, ignore_index=True)
    print(f"✅ Weather combined: {len(weather_combined):,} records")
    
    # 2. UV time-scaling factors
    UV_PERIOD_FACTORS = {
        'morning_early': 0.0, 'morning_late': 0.0,
        'noon_early': 0.3, 'noon_late': 0.8,
        'afternoon_early': 1.0, 'afternoon_late': 0.7,
        'evening_early': 0.2, 'evening_late': 0.0
    }
    
    # 3. Combine UV data
    uv_combined = pd.concat(all_uv_data, ignore_index=True)
    print(f"✅ UV combined: {len(uv_combined):,} records")
    
    # 4. Combine Air Quality data
    aq_combined = pd.concat(all_aq_data, ignore_index=True)
    print(f"✅ Air Quality combined: {len(aq_combined):,} records")
    
    # Merge Weather + UV
    weather_combined['date'] = pd.to_datetime(weather_combined['date']).dt.date
    uv_combined['date'] = pd.to_datetime(uv_combined['date']).dt.date
    
    merged_df = weather_combined.merge(
        uv_combined[['city_id', 'date', 'uv_index']],
        on=['city_id', 'date'],
        how='left'
    )
    
    # Apply UV time-scaling
    merged_df['uv_index'] = merged_df.apply(
        lambda row: row['uv_index'] * UV_PERIOD_FACTORS.get(row['period'], 0) if pd.notna(row['uv_index']) else None,
        axis=1
    )
    
    # Merge Air Quality
    aq_combined['date'] = pd.to_datetime(aq_combined['date']).dt.date
    
    final_df = merged_df.merge(
        aq_combined[['city_id', 'date', 'period', 'aqi', 'pm2_5', 'pm10', 'co', 'no2', 'o3', 'so2', 'nh3']],
        on=['city_id', 'date', 'period'],
        how='left'
    )
    
    # Add metadata
    final_df['date'] = pd.to_datetime(final_df['date'])
    final_df['year'] = final_df['date'].dt.year
    final_df['month'] = final_df['date'].dt.month
    final_df['day'] = final_df['date'].dt.day
    
    print(f"\n✅ Final merged dataset: {len(final_df):,} records")
    print(f"   Date range: {final_df['date'].min()} to {final_df['date'].max()}")
    print(f"   Cities: {final_df['city_name'].nunique()}")
    
    # Save raw data
    final_df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8')
    print(f"\n💾 Raw data saved: {OUTPUT_CSV}")
else:
    print("⏭️ Skipped merging (no crawled data). Will load from uploaded CSV.")
    final_df = None

## Bước 6: OPTION 2 - Upload CSV (nếu không crawl)

Nếu bạn đã có file `weather_raw_2022_2024.csv` từ local, upload ở đây thay vì crawl lại.

In [ ]:
# OPTION 2: Upload file CSV từ local (nếu đã crawl sẵn)
if final_df is None:
    from google.colab import files
    
    print("📤 Upload file weather_raw_2022_2024.csv:")
    uploaded = files.upload()
    
    if 'weather_raw_2022_2024.csv' in uploaded:
        final_df = pd.read_csv('weather_raw_2022_2024.csv')
        final_df['date'] = pd.to_datetime(final_df['date'])
        print(f"✅ Loaded {len(final_df):,} records")
        print(f"   Date range: {final_df['date'].min()} to {final_df['date'].max()}")
        print(f"   Cities: {final_df['city_name'].nunique()}")
    else:
        print("❌ No file uploaded.")
        final_df = None
else:
    print("✅ Already have data from crawling. Skip upload.")

## Bước 7: Label dữ liệu với WHO risk levels

In [ ]:
if final_df is not None:
    print("🏷️ Labeling data...")
    final_df['health_risk_score'] = final_df.apply(calculate_health_risk_score, axis=1)
    final_df['risk_level'] = final_df['health_risk_score'].apply(assign_risk_level)
    
    risk_counts = final_df['risk_level'].value_counts().sort_index()
    print("\n📊 Risk level distribution (BEFORE balancing):")
    print(risk_counts)
    print(f"\nPercentage:")
    print((risk_counts / len(final_df) * 100).round(2))
    
    # Visualize
    plt.figure(figsize=(12, 6))
    risk_colors = {'safe': 'green', 'low': 'lightgreen', 'moderate': 'yellow', 'high': 'orange', 'extreme': 'red'}
    ax = risk_counts.plot(kind='bar', color=[risk_colors.get(x, 'gray') for x in risk_counts.index])
    plt.title('Risk Level Distribution - BEFORE Balancing', fontsize=14, fontweight='bold')
    plt.xlabel('Risk Level')
    plt.ylabel('Count')
    plt.xticks(rotation=0)
    plt.grid(axis='y', alpha=0.3)
    for i, v in enumerate(risk_counts):
        pct = v / len(final_df) * 100
        ax.text(i, v, f'{pct:.1f}%', ha='center', va='bottom', fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    print("\n⚠️ CLASS IMBALANCE DETECTED!" if risk_counts.max() / risk_counts.min() > 5 else "\n✅ Classes relatively balanced")
else:
    print("❌ No data to label. Upload CSV first.")

## Bước 8: Feature Engineering

In [ ]:
if final_df is not None:
    print("🔧 Feature engineering...")
    
    # Daily min/max temperatures
    daily_stats = final_df.groupby(['city_id', 'date'])['temperature'].agg(['min', 'max']).reset_index()
    daily_stats.columns = ['city_id', 'date', 'temp_min', 'temp_max']
    final_df = final_df.merge(daily_stats, on=['city_id', 'date'], how='left')
    
    # Derived features
    final_df['temp_range'] = final_df['temp_max'] - final_df['temp_min']
    final_df['is_hot'] = (final_df['temperature'] > 32).astype(int)
    final_df['is_cold'] = (final_df['temperature'] < 18).astype(int)
    final_df['is_humid'] = (final_df['humidity'] > 70).astype(int)
    final_df['is_dry'] = (final_df['humidity'] < 40).astype(int)
    final_df['uv_high'] = (final_df['uv_index'].fillna(0) > 6).astype(int)
    final_df['has_rain'] = (final_df['precipitation'] > 5).astype(int)
    final_df['pm25_high'] = (final_df['pm2_5'].fillna(0) > 25).astype(int)
    final_df['aqi_poor'] = (final_df['aqi'].fillna(1) >= 4).astype(int)
    
    print("✅ Features created")
    
    # Feature columns
    FEATURE_COLS = [
        'temperature', 'temp_min', 'temp_max', 'temp_range',
        'humidity', 'precipitation', 'wind_speed',
        'uv_index', 'pm2_5', 'pm10', 'aqi',
        'co', 'no2', 'o3', 'so2', 'nh3',
        'is_hot', 'is_cold', 'is_humid', 'is_dry',
        'uv_high', 'has_rain', 'pm25_high', 'aqi_poor'
    ]
    
    # Fill missing values
    print("🔧 Filling missing values...")
    for col in FEATURE_COLS:
        if col in final_df.columns:
            final_df[col] = final_df[col].fillna(final_df[col].median())
    
    print("✅ Missing values filled")
else:
    print("❌ No data. Upload CSV first.")

## Bước 9: Apply SMOTE + Under-sampling

In [ ]:
if final_df is not None and 'FEATURE_COLS' in locals():
    print("⚙️ Applying SMOTE + Under-sampling...")
    
    X = final_df[FEATURE_COLS].copy()
    y = final_df['risk_level'].copy()
    
    # Encode labels
    le = LabelEncoder()
    y_encoded = le.fit_transform(y)
    
    print("\n📊 Original distribution:")
    original_dist = Counter(y)
    for label, count in sorted(original_dist.items()):
        print(f"   {label}: {count:,} ({count/len(y)*100:.2f}%)")
    
    # Auto-detect available classes
    available_classes = sorted(original_dist.keys())
    print(f"\n⚙️ Detected classes: {available_classes}")
    
    # Find baseline (use 'moderate' if exists, else median)
    if 'moderate' in available_classes:
        baseline_count = original_dist['moderate']
        baseline_class = 'moderate'
    else:
        counts = sorted(original_dist.values())
        baseline_count = counts[len(counts)//2]
        baseline_class = [k for k, v in original_dist.items() if v == baseline_count][0]
    
    print(f"   Baseline: {baseline_class} ({baseline_count:,} records)")
    
    # Define target ratios (only for classes that exist)
    target_ratios = {
        'safe': 0.6,
        'low': 0.8,
        'moderate': 1.0,
        'high': 0.8,
        'extreme': 0.4
    }
    
    # Only include classes that exist in data
    target_counts = {}
    for cls in available_classes:
        if cls in target_ratios:
            target_counts[cls] = int(baseline_count * target_ratios[cls])
        else:
            target_counts[cls] = original_dist[cls]
    
    print("\n🎯 Target distribution:")
    for label, count in sorted(target_counts.items()):
        print(f"   {label}: {count:,}")
    
    # Apply SMOTE + Under-sampling
    sampling_strategy = {le.transform([label])[0]: count for label, count in target_counts.items()}
    
    smote = SMOTE(sampling_strategy='auto', random_state=42, k_neighbors=5)
    undersample = RandomUnderSampler(sampling_strategy=sampling_strategy, random_state=42)
    
    X_smote, y_smote = smote.fit_resample(X, y_encoded)
    X_balanced, y_balanced = undersample.fit_resample(X_smote, y_smote)
    
    y_balanced_labels = le.inverse_transform(y_balanced)
    
    print("\n✅ Balancing completed!")
    print("\n📊 Balanced distribution:")
    balanced_dist = Counter(y_balanced_labels)
    for label, count in sorted(balanced_dist.items()):
        print(f"   {label}: {count:,} ({count/len(y_balanced_labels)*100:.2f}%)")
    
    # Visualize comparison
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    risk_colors = {'safe': 'green', 'low': 'lightgreen', 'moderate': 'yellow', 'high': 'orange', 'extreme': 'red'}
    
    axes[0].bar(original_dist.keys(), original_dist.values(), color=[risk_colors.get(k, 'gray') for k in original_dist.keys()])
    axes[0].set_title('BEFORE Balancing', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Risk Level')
    axes[0].set_ylabel('Count')
    axes[0].grid(axis='y', alpha=0.3)
    
    axes[1].bar(balanced_dist.keys(), balanced_dist.values(), color=[risk_colors.get(k, 'gray') for k in balanced_dist.keys()])
    axes[1].set_title('AFTER Balancing', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Risk Level')
    axes[1].set_ylabel('Count')
    axes[1].grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print("❌ No data or features. Complete previous steps first.")

## Bước 10: Lưu Balanced Dataset

In [ ]:
if 'X_balanced' in locals() and 'y_balanced_labels' in locals():
    print("💾 Saving balanced dataset...")
    
    balanced_df = pd.DataFrame(X_balanced, columns=FEATURE_COLS)
    balanced_df['risk_level'] = y_balanced_labels
    
    def reverse_risk_score(risk_level):
        if risk_level == "safe": return 1
        elif risk_level == "low": return 4
        elif risk_level == "moderate": return 7
        elif risk_level == "high": return 10
        else: return 14
    
    balanced_df['health_risk_score'] = balanced_df['risk_level'].apply(reverse_risk_score)
    
    # Add metadata (synthetic)
    balanced_df['city_id'] = 0
    balanced_df['city_name'] = 'Synthetic'
    balanced_df['date'] = pd.Timestamp.now().date()
    balanced_df['year'] = 0
    balanced_df['month'] = 0
    balanced_df['day'] = 0
    balanced_df['period'] = 'unknown'
    
    # Save to CSV
    balanced_df.to_csv(BALANCED_CSV, index=False, encoding='utf-8')
    
    print(f"\n✅ Balanced dataset saved: {BALANCED_CSV}")
    print(f"   Total records: {len(balanced_df):,}")
    print(f"   Features: {len(FEATURE_COLS)}")
    print(f"\n📥 Download file:")
    files.download(BALANCED_CSV)
else:
    print("❌ No balanced data to save. Complete balancing step first.")

## 📋 Tóm tắt

### ✅ Kết quả:

**1. Raw Dataset (3 years):**
- File: `weather_raw_2022_2024.csv`
- Records: ~552,000 (63 cities × 1,096 days × 8 periods)
- Features: 13 (weather + UV + air quality)
- Class distribution: IMBALANCED (safe >> extreme)

**2. Balanced Dataset:**
- File: `weather_balanced_2022_2024.csv`
- Records: ~300,000-400,000 (tuỳ SMOTE ratio)
- Class distribution: BALANCED (moderate ≈ low ≈ high)
- Ready cho XGBoost training!

### 📊 Class Balancing Strategy:
- **SMOTE**: Tạo synthetic samples cho minority classes
- **Under-sampling**: Giảm majority classes
- **Target ratio**: moderate (100%) → low/high (80%) → safe (60%) → extreme (40%)

### 🎯 Tiếp theo:
1. Download file `weather_balanced_2022_2024.csv`
2. Train XGBoost model với balanced data
3. So sánh accuracy với model cũ (unbalanced)
4. Evaluate trên real test set

### ⚠️ Lưu ý:
- Metadata (city_id, date) là synthetic (do SMOTE)
- Chỉ dùng balanced data cho TRAINING
- Test set phải dùng real data